<a href="https://colab.research.google.com/github/f1725developmenttechnologies-create/klarixa-ecosistema-ia/blob/main/Klarixa_ecosistema_ia%F0%9F%A7%A01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import subprocess
import sys
import asyncio

packages = ["fastapi", "uvicorn", "pyngrok", "groq", "pydantic", "nest_asyncio"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)

import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from groq import Groq

nest_asyncio.apply()

# 🔑 Lee las llaves desde el entorno de ejecucion (o las deja vacias para pasar parametros)
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "")

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

app = FastAPI(title="KLARIXA Brain 1 - Reasoning Engine")

class ReasoningPayload(BaseModel):
    prompt: str
    system_instruction: str = "Eres KLARIXA Brain 1, especialista en razonamiento logico y sintesis."

@app.get("/health")
async def health():
    return {"node": "Brain 1 (Groq)", "status": "online"}

@app.post("/process")
async def process_logic(payload: ReasoningPayload):
    if not groq_client:
        raise HTTPException(status_code=500, detail="GROQ_API_KEY no configurada")
    try:
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": payload.system_instruction},
                {"role": "user", "content": payload.prompt},
            ],
            temperature=0.5
        )
        return {
            "status": "success",
            "brain": "Brain 1 (openai/gpt-oss-120b)",
            "output": response.choices[0].message.content
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

port = 8001
ngrok.kill()
if NGROK_AUTHTOKEN:
    public_url = ngrok.connect(port)
    print("="*60)
    print(f"🚀 BRAIN 1 PUBLIC URL: {public_url.public_url}")
    print(f"🔗 Healthcheck: {public_url.public_url}/health")
    print("="*60)

config = uvicorn.Config(app, host="0.0.0.0", port=port, loop="asyncio")
server = uvicorn.Server(config)
asyncio.create_task(server.serve())

<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.13/dist-packages/uvicorn/server.py:79>>